# 🚀 Moon Lander - Deep Q-Network (DQN) Training
This notebook trains an RL agent to land a spacecraft on the moon using Deep Q-Learning with periodic checkpointing.

## 1. Setup and Imports

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque, namedtuple
import random
from tqdm import tqdm
import os
from datetime import datetime

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'gymnasium'

## 2. Define the DQN Neural Network

In [ ]:
class DQN(nn.Module):
    """Deep Q-Network for LunarLander"""
    
    def __init__(self, state_size, action_size, hidden_size=128):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, action_size)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

print("✓ DQN network defined")

## 3. Experience Replay Buffer

In [ ]:
Experience = namedtuple('Experience', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    """Fixed-size buffer to store experience tuples"""
    
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)
    
    def add(self, state, action, reward, next_state, done):
        experience = Experience(state, action, reward, next_state, done)
        self.buffer.append(experience)
    
    def sample(self, batch_size):
        experiences = random.sample(self.buffer, batch_size)
        
        states = torch.FloatTensor(np.array([e.state for e in experiences])).to(device)
        actions = torch.LongTensor(np.array([e.action for e in experiences])).to(device)
        rewards = torch.FloatTensor(np.array([e.reward for e in experiences])).to(device)
        next_states = torch.FloatTensor(np.array([e.next_state for e in experiences])).to(device)
        dones = torch.FloatTensor(np.array([e.done for e in experiences])).to(device)
        
        return states, actions, rewards, next_states, dones
    
    def __len__(self):
        return len(self.buffer)

print("✓ Replay buffer defined")

## 4. DQN Agent with Periodic Checkpointing

In [ ]:
class DQNAgent:
    """DQN Agent with experience replay and target network"""
    
    def __init__(self, state_size, action_size, lr=1e-3, gamma=0.99, 
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Q-Network and Target Network
        self.qnetwork = DQN(state_size, action_size).to(device)
        self.target_network = DQN(state_size, action_size).to(device)
        self.target_network.load_state_dict(self.qnetwork.state_dict())
        
        self.optimizer = optim.Adam(self.qnetwork.parameters(), lr=lr)
        self.memory = ReplayBuffer()
        
    def act(self, state, train=True):
        """Select action using epsilon-greedy policy"""
        if train and random.random() < self.epsilon:
            return random.randrange(self.action_size)
        
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        self.qnetwork.eval()
        with torch.no_grad():
            action_values = self.qnetwork(state)
        self.qnetwork.train()
        return action_values.argmax().item()
    
    def step(self, state, action, reward, next_state, done):
        """Save experience and learn from batch"""
        self.memory.add(state, action, reward, next_state, done)
    
    def learn(self, batch_size=64):
        """Update Q-network using batch of experiences"""
        if len(self.memory) < batch_size:
            return None
        
        states, actions, rewards, next_states, dones = self.memory.sample(batch_size)
        
        # Get current Q values
        current_q = self.qnetwork(states).gather(1, actions.unsqueeze(1))
        
        # Get target Q values
        next_q = self.target_network(next_states).max(1)[0].detach()
        target_q = rewards + (self.gamma * next_q * (1 - dones))
        
        # Compute loss
        loss = F.mse_loss(current_q.squeeze(), target_q)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
        
        return loss.item()
    
    def update_target_network(self):
        """Copy weights from Q-network to target network"""
        self.target_network.load_state_dict(self.qnetwork.state_dict())
    
    def save_checkpoint(self, filepath, episode, scores):
        """Save model checkpoint"""
        checkpoint = {
            'episode': episode,
            'qnetwork_state_dict': self.qnetwork.state_dict(),
            'target_network_state_dict': self.target_network.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'epsilon': self.epsilon,
            'scores': scores
        }
        torch.save(checkpoint, filepath)
        print(f"✓ Checkpoint saved: {filepath}")
    
    def load_checkpoint(self, filepath):
        """Load model checkpoint"""
        checkpoint = torch.load(filepath, map_location=device)
        self.qnetwork.load_state_dict(checkpoint['qnetwork_state_dict'])
        self.target_network.load_state_dict(checkpoint['target_network_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.epsilon = checkpoint['epsilon']
        print(f"✓ Checkpoint loaded: {filepath}")
        return checkpoint['episode'], checkpoint['scores']

print("✓ DQN agent defined")

## 5. Training Configuration

In [ ]:
# Training hyperparameters
EPISODES = 1000              # Total number of episodes
MAX_STEPS = 1000             # Max steps per episode
BATCH_SIZE = 64              # Batch size for learning
TARGET_UPDATE_FREQ = 10      # Update target network every N episodes
CHECKPOINT_FREQ = 100        # Save checkpoint every N episodes
PRINT_FREQ = 10              # Print progress every N episodes

# Create checkpoint directory
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('videos', exist_ok=True)

print("✓ Configuration set")
print(f"  Episodes: {EPISODES}")
print(f"  Checkpoint frequency: Every {CHECKPOINT_FREQ} episodes")

## 6. Initialize Environment and Agent

In [ ]:
# Create environment
env = gym.make('LunarLander-v3')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print(f"Environment: LunarLander-v3")
print(f"  State size: {state_size}")
print(f"  Action size: {action_size}")
print(f"  Actions: 0=do nothing, 1=fire left, 2=fire main, 3=fire right")

# Create agent
agent = DQNAgent(state_size, action_size)
print("\n✓ Agent initialized")

## 7. Training Loop with Periodic Checkpointing

In [ ]:
def train_agent(agent, env, episodes=EPISODES, load_checkpoint_path=None):
    """Train the DQN agent with periodic checkpointing"""
    
    scores = []
    avg_scores = []
    start_episode = 0
    
    # Load checkpoint if provided
    if load_checkpoint_path and os.path.exists(load_checkpoint_path):
        start_episode, scores = agent.load_checkpoint(load_checkpoint_path)
        print(f"Resuming from episode {start_episode}")
    
    # Training loop
    for episode in tqdm(range(start_episode, episodes), desc="Training"):
        state, _ = env.reset()
        score = 0
        
        for step in range(MAX_STEPS):
            # Select and perform action
            action = agent.act(state, train=True)
            next_state, reward, done, truncated, _ = env.step(action)
            
            # Store experience and learn
            agent.step(state, action, reward, next_state, done or truncated)
            agent.learn(BATCH_SIZE)
            
            state = next_state
            score += reward
            
            if done or truncated:
                break
        
        scores.append(score)
        avg_score = np.mean(scores[-100:])
        avg_scores.append(avg_score)
        
        # Update target network
        if episode % TARGET_UPDATE_FREQ == 0:
            agent.update_target_network()
        
        # Print progress
        if episode % PRINT_FREQ == 0:
            print(f"\nEpisode {episode}/{episodes} | Score: {score:.2f} | Avg: {avg_score:.2f} | Epsilon: {agent.epsilon:.3f}")
        
        # Save periodic checkpoint
        if episode % CHECKPOINT_FREQ == 0 and episode > 0:
            checkpoint_path = f"checkpoints/checkpoint_ep{episode}.pth"
            agent.save_checkpoint(checkpoint_path, episode, scores)
        
        # Early stopping if solved
        if avg_score >= 200:
            print(f"\n🎉 Environment solved in {episode} episodes! Avg Score: {avg_score:.2f}")
            agent.save_checkpoint(f"checkpoints/solved_ep{episode}.pth", episode, scores)
            break
    
    # Save final checkpoint
    agent.save_checkpoint(f"checkpoints/final_ep{episodes}.pth", episodes, scores)
    
    return scores, avg_scores

print("✓ Training function ready")

## 8. Start Training

In [ ]:
# Start training (or resume from checkpoint)
print("🚀 Starting training...\n")

# To resume training, uncomment and specify checkpoint:
# scores, avg_scores = train_agent(agent, env, load_checkpoint_path='checkpoints/checkpoint_ep100.pth')

scores, avg_scores = train_agent(agent, env)

print("\n✅ Training complete!")

## 9. Visualize Training Progress

In [ ]:
# Plot training progress
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(scores, alpha=0.3, label='Episode Score')
ax.plot(avg_scores, linewidth=2, label='Average Score (100 episodes)')
ax.axhline(y=200, color='r', linestyle='--', label='Solved Threshold')
ax.set_xlabel('Episode')
ax.set_ylabel('Score')
ax.set_title('Moon Lander Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_progress.png', dpi=150)
plt.show()

print(f"Best average score: {max(avg_scores):.2f}")
print(f"Final average score: {avg_scores[-1]:.2f}")

## 10. Test Trained Agent

In [ ]:
def test_agent(agent, env, episodes=5, render=False):
    """Test the trained agent"""
    test_scores = []
    
    for episode in range(episodes):
        state, _ = env.reset()
        score = 0
        done = False
        
        while not done:
            action = agent.act(state, train=False)  # No exploration
            next_state, reward, done, truncated, _ = env.step(action)
            state = next_state
            score += reward
            done = done or truncated
        
        test_scores.append(score)
        print(f"Test Episode {episode + 1}: Score = {score:.2f}")
    
    avg_test_score = np.mean(test_scores)
    print(f"\nAverage Test Score: {avg_test_score:.2f}")
    return test_scores

# Test the agent
print("Testing trained agent...\n")
test_scores = test_agent(agent, env, episodes=5)

env.close()

## 11. Visualize Agent Performance

In [ ]:
# Create a render environment to record video
render_env = gym.make('LunarLander-v3', render_mode='rgb_array')

def record_episode(agent, env, filename='videos/landing.gif'):
    """Record a video of the agent landing"""
    import imageio
    
    frames = []
    state, _ = env.reset()
    done = False
    score = 0
    
    while not done:
        frames.append(env.render())
        action = agent.act(state, train=False)
        state, reward, done, truncated, _ = env.step(action)
        score += reward
        done = done or truncated
    
    # Save as GIF
    imageio.mimsave(filename, frames, fps=30)
    print(f"✓ Video saved: {filename} (Score: {score:.2f})")
    return score

# Record a landing
print("Recording landing video...")
score = record_episode(agent, render_env)
render_env.close()

print("\n🎬 You can view the landing video at: videos/landing.gif")

## 12. Load and Continue Training from Checkpoint

In [ ]:
# Example: Continue training from a checkpoint
# Uncomment to use:

# new_agent = DQNAgent(state_size, action_size)
# new_env = gym.make('LunarLander-v3')
# 
# # Resume from checkpoint
# scores, avg_scores = train_agent(
#     new_agent, 
#     new_env, 
#     episodes=2000,  # Train for more episodes
#     load_checkpoint_path='checkpoints/checkpoint_ep100.pth'
# )

print("To resume training, uncomment the code above and run this cell")